# clue_fix

> Phase 13 options on the clue board, called from `TextClueEnv`: requests (`ask`), the fact table (`table`), senders' track records (`record`) and costly verification (`check`): their rule sentences, observation lines, parsing and the schema; all off, the Phase 11 interface is unchanged.

In [ ]:
#| default_exp clue_fix

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy
import re

from explRL.clue_agents import ClueBoard, known

FIX = ('ask', 'table', 'record', 'check')
FIX_RULES = dict(
    ask="'ask' for the clues you lack on the subtask you have open (target = it): your request shows on the board to "
        "every teammate until you open another subtask or it closes; asking takes your turn.",
    table='The board lists each clue known to the team once per subtask.',
    record="Each shared clue shows how many of its sender's earlier shares were true, counted over subtasks solved so far.",
    check="'check' a clue on the board (target = its subtask, clue = its text): next turn you learn whether it is true; "
          "checking takes your turn.")
_RE = re.compile(r'x\s*mod\s*(\d+)\s*=\s*(-?\d+)', re.I)


def fixOptions(world):
    "The options set in a cfg.world, as TextClueEnv keywords (absent: off)."
    return {k: world[k] for k in FIX if k in world}


def fixSchema(schema, ask=False, check=False):
    "The schema itself with both actions off (Phase 11 requests unchanged); else a copy whose action enum adds them."
    if not (ask or check):
        return schema
    s = copy.deepcopy(schema)
    s['properties']['action']['enum'] += ['ask'] * bool(ask) + ['check'] * bool(check)
    return s


def trackRecord(a, board):
    "(true, checked) over agent a's shares of the subtasks solved so far (a solved subtask's clues are known)."
    s = [not x[6] for x in board.shareLog if x[1] == a and board.tasks[x[2]].solvedBy >= 0]
    return sum(s), len(s)


def _list(xs, word):
    xs = [f"'{x}'" for x in xs]
    return ', '.join(xs[:-1]) + f' {word} ' + xs[-1] if len(xs) > 1 else xs[0]


def fixRules(env, text):
    "The rules with the options' sentences after the channel paragraph and the new actions in the reply format."
    on, new = [k for k in FIX if getattr(env, k)], [k for k in ('ask', 'check') if getattr(env, k)]
    if not on:
        return text
    text = text.replace('did not see it.\n\n', 'did not see it.\n\n' + ' '.join(FIX_RULES[k] for k in on) + '\n\n', 1)
    if new:
        text = text.replace("action is 'open', 'submit', 'share' or 'wait'",
                            'action is ' + _list(['open', 'submit', 'share', 'wait'] + new, 'or'))
        text = text.replace("subtask name for 'open' and 'share'", 'subtask name for ' + _list(['open', 'share'] + new, 'and'))
        text = text.replace("clue text for 'share' (", 'clue text for ' + _list(['share'] + ['check'] * env.check, 'and') + ' (')
    return text


def fixBoard(env, t, view):
    "The board line with `table` (each distinct clue once per subtask, no sender) or `record` (senders' track records)."
    if env.table:
        return 'Known to the team: ' + ('; '.join(f'T{k}: ' + ', '.join(dict.fromkeys(
            f'x mod {p} = {r}' for _, p, r, _, _ in sorted(xs, key=lambda x: x[4]))) for k, xs in view.items()) or 'none')
    rec = {a: trackRecord(a, env.world) for a in {x[3] for xs in view.values() for x in xs}}
    tr = lambda a: f'record {rec[a][0]}/{rec[a][1]} true' if rec[a][1] else 'record none'
    return 'Clues shared on the board: ' + ('; '.join(f'T{k}: x mod {p} = {r} (agent {a}, {t - ts:g} turns ago, {tr(a)})'
                                                      for k, xs in view.items() for _, p, r, a, ts in xs) or 'none')


def fixLines(env, i, t, channel):
    "Observation lines of the options: the live requests (`ask`) and the result of the agent's last check (`check`)."
    b, out, mine = env.world, [], env._known.get(i, {})
    if env.ask and isinstance(channel, ClueBoard):
        def lack(k, a):
            theirs = known(a, b, channel, env.checked.get(a, {}) if env.check else None)[k]
            return sum(j not in theirs for j in mine.get(k, {}))
        out.append('Requests: ' + ('; '.join(f'agent {a} needs clues for T{k} (you hold {lack(k, a)})'
                                             for k, a, _ in channel.requests(b)) or 'none'))
    if i in env._checks:
        out.append(env._checks.pop(i))
    return out


def fixParse(env, parsed, i, a, k):
    """'ask' (target = the agent's open subtask, still open) and 'check' (target open, clue = a clue posted for it on
    the board) -> a wait on the board, with report ('ask', k) for an ask; anything else invalid."""
    b, ch, wait = env.world, env._chan, ('wait', None)
    if not isinstance(ch, ClueBoard) or k not in b.open:
        return wait, None, False
    if a == 'ask' and env.ask and b.opened[i] == k:
        env._did[i] = f'asked for clues for T{k}'
        return wait, ('ask', k), True
    m = _RE.search(str(parsed.get('clue', '')))
    posts = sorted((x for x in ch.posts.get(k, []) if m and (x[1], x[2]) == (int(m.group(1)), int(m.group(2)))),
                   key=lambda x: x[4])
    if a == 'check' and env.check and posts:
        j, p, r, src, _ = posts[0]                     # the sender shown is the clue's first poster
        ok = b.tasks[k].task.clue(j)[1] == r
        env.checked.setdefault(i, {})[(k, j, r)] = ok
        env._checks[i] = f"Your check: T{k} x mod {p} = {r} (from agent {src}) is {'true' if ok else 'false'}."
        env._did[i] = f'checked T{k}: x mod {p} = {r}'
        return wait, None, True
    return wait, None, False

## Tests

In [ ]:
import hashlib
import json
from explRL.clue_fix import *
from explRL.clue_agents import ClueBoard, known
from explRL.clue_runs import runClue
from explRL.clue_text import CLUE_SCHEMA, HINTS, TextClueEnv
from explRL.clueboard import ClueBoardEnv
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig

# defaults unchanged (AC-2, NFR-1): rules, schema, observations, rows and turn logs hash as before Phase 13
H = lambda s: hashlib.sha256(s.encode()).hexdigest()[:16]
fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
got = [H(TextClueEnv(ClueBoardEnv(0, 4)).rules()), H(TextClueEnv(ClueBoardEnv(1, 8, m=2, R=30), hint='follow').rules()),
       H(json.dumps(CLUE_SCHEMA, sort_keys=True))]
for ch, nLiar in (('blackboard', 1), ('messages', 0), ('hierarchical', 0)):
    cfg = LLMConfig(task='clue', channel=ch, N=4, turns=10, nLiar=nLiar, memory='full', world=dict(m=2, r=1, R=30))
    got.append(H(json.dumps(runClue(cfg, 900, fake()), sort_keys=True, default=str)))
assert got == ['0d71b754be52dbba', '78a7b077e3f81a06', 'ce192d3fc3c7e717', 'f35507dc0199d7e3', 'a0e448242d25f405',
               'b131cbde560b9608'], got
off = dict(ask=False, table=False, record=False, check=False)
b0 = ClueBoardEnv(0, 4)
assert TextClueEnv(b0).SCHEMA is CLUE_SCHEMA and TextClueEnv(b0, **off).SCHEMA is CLUE_SCHEMA
assert TextClueEnv(b0, table=True, record=True).SCHEMA is CLUE_SCHEMA and TextClueEnv(b0, **off).rules() == TextClueEnv(b0).rules()
assert fixOptions(dict(m=2, hint='follow')) == {} and fixOptions(dict(ask=True, R=30)) == dict(ask=True)
# the schema gains 'ask' and 'check' only with their options; CLUE_SCHEMA itself never changes
enum = lambda e: e.SCHEMA['properties']['action']['enum']
assert enum(TextClueEnv(b0, ask=True)) == ['open', 'submit', 'share', 'wait', 'ask']
assert enum(TextClueEnv(b0, check=True)) == ['open', 'submit', 'share', 'wait', 'check']
assert enum(TextClueEnv(b0, ask=True, check=True)) == ['open', 'submit', 'share', 'wait', 'ask', 'check']
assert CLUE_SCHEMA['properties']['action']['enum'] == ['open', 'submit', 'share', 'wait']
# rules: each option adds its sentence after the channel paragraph; ask and check join the reply format; the hints
base = TextClueEnv(b0).rules()
for k in FIX:
    r = TextClueEnv(b0, **{k: True}).rules()
    assert FIX_RULES[k] in r and r.index(FIX_RULES[k]) > r.index('did not see it.') and r.replace(FIX_RULES[k] + '\n\n', '').split(
        'Reply with JSON')[0] == base.split('Reply with JSON')[0], k
r = TextClueEnv(b0, ask=True, check=True).rules()
assert "action is 'open', 'submit', 'share', 'wait', 'ask' or 'check'; target is the subtask name for 'open', 'share', 'ask' " \
       "and 'check' (empty otherwise)" in r and "clue is the clue text for 'share' and 'check' (empty" in r
assert "action is 'open', 'submit', 'share', 'wait' or 'ask';" in TextClueEnv(b0, ask=True).rules()
assert TextClueEnv(b0, table=True).rules().split('Reply with JSON')[1] == base.split('Reply with JSON')[1]
for h in ('evidence', 'warn'):
    assert HINTS[h] in TextClueEnv(b0, hint=h).rules() and TextClueEnv(b0, hint=h).rules().replace(HINTS[h] + '\n\n', '') == base
assert HINTS['evidence'].startswith('Before working on your own subtask, post every clue') and 'Cross-check' in HINTS['warn']

P = lambda **kw: dict(dict(action='wait', target='', answer=0, clue='', notes=''), **kw)
def clue(b, k, j):
    p, r = b.tasks[k].task.clue(j)
    return j, p, r
def share(b, cb, i, k, j, r, t):
    "Agent i posts clue j of k with residue r (true or false) on the board and the board logs the share."
    b.apply(i, ('share', (k, j, r)), t); cb.post(i, k, (j, b.tasks[k].task.primes[j], r), t)
def solve(b, i, k, t):
    b.apply(i, ('open', k), t); assert b.apply(i, ('submit', b.tasks[k].task.x), t + 1)[3]['result'] == 'solved'

# ask: valid only for the asker's open subtask (still open); a wait on the board; posted, shown with holder counts; lapses
b, cb = ClueBoardEnv(0, 4, m=3, r=1, R=105, life=100), ClueBoard()
env, plain = TextClueEnv(b, ask=True), TextClueEnv(b)
k, k2 = b.open[0], b.open[1]
A = next(a for a in range(4) if len(b.held(a, k)) < 3)          # the asker lacks a clue of k
V = next(a for a in range(4) if a != A and any(j not in b.held(A, k) for j in b.held(a, k)))   # a viewer holding one
b.apply(A, ('open', k), 0.0)
assert 'Requests: none' in env.observe(A, 1.0, cb) and 'Requests' not in plain.observe(A, 1.0, cb)
assert env.parse(P(action='ask', target=f'T{k2}'), A) == (('wait', None), None, False)          # not its open subtask
assert env.parse(P(action='ask', target=f'T{k}'), V)[2] is False                                # V has nothing open
assert plain.parse(P(action='ask', target=f'T{k}'), A) == (('wait', None), None, False)          # option off: invalid
act, rep, ok = env.parse(P(action='ask', target=f'T{k}'), A)
assert (act, rep, ok) == (('wait', None), ('ask', k), True)
_, _, _, info = b.apply(A, act, 1.0)
assert info['result'] == 'wait' and b.ledger[-1][3] == 'stay' and b.counts == dict(solved=0, wrong=0, late=0, opens=1, shares=0)
assert env.post(A, rep, 1.0, cb, info) and cb.requests(b) == [(k, A, 1.0)]
assert env.summary(A, 1.0, act, 0.0, rep, info) == f'turn 1: asked for clues for T{k}, earned 0.00.'
n = sum(j not in b.held(A, k) for j in b.held(V, k))
assert f'Requests: agent {A} needs clues for T{k} (you hold {n})' in env.observe(V, 2.0, cb)
jv = next(j for j in b.held(V, k) if j not in b.held(A, k))
share(b, cb, V, k, *clue(b, k, jv)[::2], 2.0)                      # once posted, the asker knows it: one fewer
assert f'agent {A} needs clues for T{k} (you hold {n - 1})' in env.observe(V, 3.0, cb)
env.observe(A, 3.0, cb); cb.ask(A, k, 3.0)                         # a new ask replaces the earlier one
assert cb.requests(b) == [(k, A, 3.0)]
B = next(a for a in range(4) if a not in (A, V))
b.apply(B, ('open', k2), 3.0); env.observe(B, 4.0, cb); assert env.parse(P(action='ask', target=f'T{k2}'), B)[2]
cb.ask(B, k2, 4.0)
assert [x[1] for x in cb.requests(b)] == sorted([A, B])
assert f'agent {A} needs clues for T{k} (you hold ' in env.observe(V, 4.0, cb) and f'agent {B} needs clues for T{k2}' in env.observe(V, 4.0, cb)
b.apply(A, ('open', b.open[2]), 4.0); b.apply(A, ('open', k), 5.0)  # A opened another subtask: lapsed (even back on k)
assert cb.requests(b) == [(k2, B, 4.0)]
solve(b, V, k2, 6.0)                                                # k2 closed: lapsed
assert cb.requests(b) == [] and 'Requests: none' in env.observe(V, 8.0, cb)

# table: each distinct clue once per open subtask, without sender or age, instead of one entry per post
b, cb = ClueBoardEnv(0, 4, m=3, r=1, R=105, life=100), ClueBoard()
k, k2 = b.open[0], b.open[1]
(j0, p0, r0), (j1, p1, r1), (j2, p2, r2) = clue(b, k, 0), clue(b, k, 1), clue(b, k2, 2)
share(b, cb, 0, k, j0, r0, 0.0); share(b, cb, 1, k, j0, r0, 1.0); share(b, cb, 2, k, j1, r1, 1.0); share(b, cb, 3, k2, j2, r2, 2.0)
o = TextClueEnv(b, table=True).observe(0, 3.0, cb)
assert 'Clues shared on the board' not in o and 'Known to the team: ' in o
line = next(x for x in o.split('\n') if x.startswith('Known to the team'))
assert f'T{k}: x mod {p0} = {r0}, x mod {p1} = {r1}' in line and f'T{k2}: x mod {p2} = {r2}' in line and 'agent' not in line
assert line.count(f'x mod {p0} = {r0}') == 1
assert TextClueEnv(b, table=True).observe(0, 3.0, ClueBoard()).count('Known to the team: none') == 1

# record: true and false shares of solved subtasks only; 'record none' without any
b, cb = ClueBoardEnv(0, 4, m=3, r=1, R=105, life=100), ClueBoard()
k, k2, k3, k4 = b.open[:4]
_, pf, rf = clue(b, k2, 0)
share(b, cb, 1, k, *clue(b, k, 0)[::2], 0.0)                        # agent 1: a true share of k (solved below)
share(b, cb, 2, k2, 0, (rf + 1) % pf, 0.0)                          # agent 2: a false share of k2 (solved below)
share(b, cb, 2, k4, 0, (clue(b, k4, 0)[2] + 1) % clue(b, k4, 0)[1], 0.0)   # and a false one of k4 (stays open)
share(b, cb, 1, k3, *clue(b, k3, 1)[::2], 1.0)                      # shares still on the board
share(b, cb, 2, k3, *clue(b, k3, 2)[::2], 1.0)
env = TextClueEnv(b, record=True)
assert trackRecord(1, b) == trackRecord(2, b) == (0, 0) and trackRecord(0, b) == (0, 0)
assert f'(agent 1, 2 turns ago, record none)' in env.observe(0, 3.0, cb)
solve(b, 3, k, 3.0); solve(b, 3, k2, 5.0)
assert trackRecord(1, b) == (1, 1) and trackRecord(2, b) == (0, 1)      # k4's false share is not counted yet
o = env.observe(0, 7.0, cb)
assert f'x mod {clue(b, k3, 1)[1]} = {clue(b, k3, 1)[2]} (agent 1, 6 turns ago, record 1/1 true)' in o
assert '(agent 2, 6 turns ago, record 0/1 true)' in o and 'Clues shared on the board: ' in o
assert TextClueEnv(b).observe(0, 7.0, cb).replace(', record 1/1 true', '').replace(', record 0/1 true', '') == o.replace(
    ', record 1/1 true', '').replace(', record 0/1 true', '')

# check: the truth next turn; a false clue leaves the checker's known clues (falling back), a true one is kept; others unchanged
b, cb = ClueBoardEnv(0, 4, m=3, r=1, R=105, life=100), ClueBoard()
k = b.open[0]
j = 0; H0 = min(b.tasks[k].holders[j]); _, p, r = clue(b, k, j)
C, D, L = [a for a in range(4) if a != H0][:3]                       # C checks, D does not; L lies
env = TextClueEnv(b, check=True)
share(b, cb, H0, k, j, r, 0.0)                                       # the true clue first
share(b, cb, L, k, j, (r + 1) % p, 1.0)                              # then a false one, later (overwrites in known)
b.apply(C, ('open', k), 1.0)
o = env.observe(C, 2.0, cb)
assert known(C, b, cb)[k][j] == (r + 1) % p and 'Your check' not in o
assert env.parse(P(action='check', target=f'T{k}', clue=f'x mod {p} = {(r + 2) % p}'), C)[2] is False   # never posted
assert TextClueEnv(b).parse(P(action='check', target=f'T{k}', clue=f'x mod {p} = {(r + 1) % p}'), C)[2] is False
act = env.parse(P(action='check', target=f'T{k}', clue=f'x mod {p} = {(r + 1) % p}'), C)
assert act == (('wait', None), None, True) and env.checked[C] == {(k, j, (r + 1) % p): False}
b.apply(C, act[0], 2.0)
o = env.observe(C, 3.0, cb)
assert f'Your check: T{k} x mod {p} = {(r + 1) % p} (from agent {L}) is false.' in o
assert env._known[C][k][j] == r and f'x mod {p} = {r}' in o.split('Clues you know: ')[1]   # falls back to the true one
assert 'Your check' not in env.observe(C, 4.0, cb)                   # shown once, the turn after the check
assert env.observe(D, 4.0, cb) and env._known[D][k][j] == (r + 1) % p  # D's view is unchanged
assert known(C, b, cb, {(k, j, r): False, (k, j, (r + 1) % p): False})[k].get(j) is None   # both found false: unknown
env.parse(P(action='check', target=f'T{k}', clue=f'x mod {p} = {r}'), C)
assert f'Your check: T{k} x mod {p} = {r} (from agent {H0}) is true.' in env.observe(C, 5.0, cb)
share(b, cb, D, k, j, (r + 2) % p, 5.0)                              # a later post: C keeps the clue it found true
assert env.observe(C, 6.0, cb) and env._known[C][k][j] == r and known(D, b, cb)[k][j] == (r + 2) % p
assert env.summary(C, 4.0, ('wait', None), 0.0, None, dict(result='wait', task=-1)) == f'turn 4: checked T{k}: x mod {p} = {r}, earned 0.00.'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()